# 0. Configuration

In [4]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style("whitegrid")
pd.set_option("display.max_columns", None)

# 1. Load Data

In [2]:
case_intel_df = pd.read_csv('datasets/case_intelligence_baseline.csv')

In [3]:
case_intel_df

,consumer_id,y_true,risk_score,predicted_label,risk_band,latest_changed_password,max_item_count,days_since_first_transaction,email_address_age_days,email_domain_label,...,per_day_unique_billing_last4,per_month_logout,per_month_page_activity,per_month_transactions,is_tp,is_fp,is_fn,is_tn,high_transaction_velocity,many_unique_ips
0,16008,0,0.000320,0,low,False,1.0,1237.847438,1533.666613,575.0,...,-999.0,-999.0,-999.0,12.074369,0,0,0,1,0,0
1,17953,0,0.003519,0,low,False,3.0,1207.080620,1066.933394,386.0,...,1.0,-999.0,5.0,0.735138,0,0,0,1,0,0
2,16218,0,0.001908,0,low,Missing,2.0,537.175247,664.548801,386.0,...,1.0,-999.0,-999.0,4.089693,0,0,0,1,0,0
3,16411,0,0.014112,0,low,False,1.0,317.478958,126.923077,386.0,...,2.0,-999.0,-999.0,4.483389,0,0,0,1,0,0
4,1697,0,0.003370,0,low,Missing,-999.0,-999.000000,-999.000000,-999.0,...,-999.0,-999.0,-999.0,-999.000000,0,0,0,1,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3292,8217,0,0.966323,1,high,False,1.0,441.495962,257.031888,1100.0,...,-999.0,1.0,1.0,7.331166,0,1,0,0,0,0
3293,3356,0,0.005665,0,low,False,2.0,164.158927,1009.217491,64.0,...,-999.0,-999.0,-999.0,4.483389,0,0,0,1,0,0
3294,19864,0,0.019353,0,low,False,1.0,314.989689,649.766126,386.0,...,-999.0,-999.0,-999.0,7.081968,0,0,0,1,0,0
3295,7653,0,0.226455,0,low,False,1.0,238.406580,46.255889,492.0,...,-999.0,-999.0,-999.0,2.225910,0,0,0,1,0,0


# 2. Build Summary Tables

In [5]:
# 2a. Cases by risk band
cases_by_risk_band = (
    case_intel_df.groupby("risk_band", observed=False)
    .agg(
        cases=("consumer_id", "count"),
        fraud_rate=("y_true", "mean"),
        flagged_rate=("predicted_label", "mean"),
        avg_risk_score=("risk_score", "mean")
    )
    .reset_index()
)

cases_by_risk_band


,risk_band,cases,fraud_rate,flagged_rate,avg_risk_score
0,high,357,0.501401,1.00000,0.900681
1,low,2631,0.005701,0.00000,0.045602
2,medium,309,0.074434,0.33657,0.450331


In [6]:
# 2b. Cases by root cause category
cases_by_root_cause = (
    case_intel_df.groupby("root_cause_category")
    .agg(
        cases=("consumer_id", "count"),
        fraud_rate=("y_true", "mean"),
        flagged_rate=("predicted_label", "mean"),
        avg_risk_score=("risk_score", "mean")
    )
    .reset_index()
    .sort_values("cases", ascending=False)
)

cases_by_root_cause


KeyError: 'root_cause_category'

In [7]:
# 2c. Top product categories among flagged users
flagged_cases = case_intel_df[case_intel_df["predicted_label"] == 1]

top_product_categories_flagged = (
    flagged_cases["latest_item_category"]
    .value_counts(dropna=False)
    .reset_index()
)

top_product_categories_flagged.columns = ["latest_item_category", "cases"]
top_product_categories_flagged.head(15)


,latest_item_category,cases
0,Missing,172
1,Beverages,17
2,Appetizers,16
3,Sides,13
4,Entrees,9
5,Chicken,8
6,Drinks,7
7,Brandy,7
8,Desserts,6
9,Burgers,5


In [8]:
# 2d. False positives by region
false_positives = case_intel_df[
    (case_intel_df["predicted_label"] == 1) & (case_intel_df["y_true"] == 0)
]

fp_by_region = (
    false_positives.groupby("latest_delivery_address_region_label")
    .agg(
        false_positive_cases=("consumer_id", "count"),
        avg_risk_score=("risk_score", "mean")
    )
    .reset_index()
    .sort_values("false_positive_cases", ascending=False)
)

fp_by_region.head(15)


,latest_delivery_address_region_label,false_positive_cases,avg_risk_score
3,5.0,68,0.764860
31,43.0,36,0.746782
7,10.0,20,0.776618
23,33.0,20,0.790596
0,-999.0,9,0.803182
24,34.0,9,0.721424
10,15.0,8,0.827899
15,20.0,7,0.767856
4,6.0,7,0.771687
18,23.0,7,0.792445


In [9]:
# 2e. False positives by product category
fp_by_category = (
    false_positives.groupby("latest_item_category")
    .agg(
        false_positive_cases=("consumer_id", "count"),
        avg_risk_score=("risk_score", "mean")
    )
    .reset_index()
    .sort_values("false_positive_cases", ascending=False)
)

fp_by_category.head(15)


,latest_item_category,false_positive_cases,avg_risk_score
67,Missing,97,0.824726
2,Appetizers,12,0.716216
36,Entrees,7,0.677436
82,Sides,7,0.696998
7,Beverages,5,0.733163
81,Side,4,0.913243
14,Burgers,4,0.834250
74,Pasta,3,0.657992
33,Drinks,3,0.582511
30,Desserts,3,0.751522


In [11]:
reason_code_cols = [
    "high_transaction_velocity",
    "many_unique_ips"
]

def combine_reason_codes(row):
    active = [col for col in reason_code_cols if row[col] == 1]
    return ", ".join(active) if active else "no_reason_code"

case_intel_df["reason_code_combo"] = case_intel_df.apply(combine_reason_codes, axis=1)


In [12]:
# 2f. Most common reason code combinations
reason_code_combos = (
    case_intel_df["reason_code_combo"]
    .value_counts()
    .reset_index()
)

reason_code_combos.columns = ["reason_code_combo", "cases"]
reason_code_combos.head(15)


,reason_code_combo,cases
0,no_reason_code,2878
1,many_unique_ips,221
2,high_transaction_velocity,166
3,"high_transaction_velocity, many_unique_ips",32


# 3. Friction Analysis

In [13]:
# 4a. Create error flags
case_intel_df["is_tp"] = ((case_intel_df["predicted_label"] == 1) & (case_intel_df["y_true"] == 1)).astype(int)
case_intel_df["is_fp"] = ((case_intel_df["predicted_label"] == 1) & (case_intel_df["y_true"] == 0)).astype(int)
case_intel_df["is_fn"] = ((case_intel_df["predicted_label"] == 0) & (case_intel_df["y_true"] == 1)).astype(int)
case_intel_df["is_tn"] = ((case_intel_df["predicted_label"] == 0) & (case_intel_df["y_true"] == 0)).astype(int)


In [14]:
# 4b. Borderline cases: adjust thresholds if you want
case_intel_df["borderline_flag"] = (
    (case_intel_df["risk_score"] >= 0.40) & (case_intel_df["risk_score"] <= 0.60)
).astype(int)

borderline_cases = case_intel_df[case_intel_df["borderline_flag"] == 1]
borderline_cases.head()


,consumer_id,y_true,risk_score,predicted_label,risk_band,latest_changed_password,max_item_count,days_since_first_transaction,email_address_age_days,email_domain_label,email_domain_tld_label,email_username_length,num_unique_delivery_addresses,latest_delivery_address_name_length,latest_delivery_address_region_label,latest_delivery_address_fraction_vowels,latest_order_amount_usd,latest_item_category,latest_item_product_title,latest_item_quantity,latest_item_tag_count,per_week_purchase_total,per_week_unique_ips,per_week_update_account,per_week_payment_method_change,per_day_add_item_to_cart,per_day_transactions,per_day_payment_method_change,per_day_devices_per_user,per_day_purchase_total,per_day_unique_billing_last4,per_month_logout,per_month_page_activity,per_month_transactions,is_tp,is_fp,is_fn,is_tn,high_transaction_velocity,many_unique_ips,reason_code_combo,borderline_flag
34,15204,0,0.462586,0,medium,False,1.0,562.775758,241.028250,1100.0,17.0,15.0,1.0,13.0,43.0,0.282479,130.712570,Traditional Favorites,Kung Pao,1.0,1.0,52.61,2.0,4.0,5.0,-999.0,3.0,2.0,1.0,0.00,3.0,-999.0,-999.0,4.983103,0,0,0,1,0,0,no_reason_code,1
50,6010,0,0.410164,0,medium,False,1.0,314.989689,161.051493,386.0,17.0,9.0,1.0,18.0,3.0,0.289665,72.369672,World-Famous Pizookies®,Pizookie® Trio,1.0,1.0,0.00,4.0,1.0,0.0,-999.0,-999.0,0.0,3.0,0.00,-999.0,-999.0,-999.0,4.483389,0,0,0,1,0,1,many_unique_ips,1
61,15215,0,0.485768,0,medium,False,1.0,462.694585,574.061093,386.0,17.0,10.0,1.0,18.0,16.0,0.341909,56.405878,Missing,Missing,-999.0,-999.0,-999.00,1.0,8.0,2.0,-999.0,7.0,2.0,1.0,-999.00,2.0,-999.0,-999.0,4.483389,0,0,0,1,1,0,high_transaction_velocity,1
83,11805,0,0.509457,1,medium,False,1.0,27.053223,13.959562,1100.0,17.0,12.0,1.0,14.0,38.0,0.341829,109.667419,Combos,Chicken Tenders Combo,1.0,1.0,-999.00,1.0,10.0,0.0,-999.0,2.0,0.0,1.0,-999.00,1.0,-999.0,-999.0,4.483389,0,1,0,0,0,0,no_reason_code,1
115,15557,0,0.496873,0,medium,False,1.0,537.668760,119.093812,386.0,17.0,13.0,1.0,13.0,15.0,0.326692,60.301682,Ribs Fried Chicken By The Box,8 Pieces Of Fried Chicken,1.0,1.0,62.29,3.0,2.0,2.0,-999.0,5.0,1.0,2.0,40.37,1.0,-999.0,-999.0,4.841196,0,0,0,1,1,1,"high_transaction_velocity, many_unique_ips",1


In [15]:
# 4c. Friction by risk band
friction_by_risk_band = (
    case_intel_df.groupby("risk_band", observed=False)
    .agg(
        cases=("consumer_id", "count"),
        false_positives=("is_fp", "sum"),
        borderline_cases=("borderline_flag", "sum")
    )
    .reset_index()
)

friction_by_risk_band["fp_rate_within_band"] = (
    friction_by_risk_band["false_positives"] / friction_by_risk_band["cases"]
)

friction_by_risk_band["borderline_rate_within_band"] = (
    friction_by_risk_band["borderline_cases"] / friction_by_risk_band["cases"]
)

friction_by_risk_band


,risk_band,cases,false_positives,borderline_cases,fp_rate_within_band,borderline_rate_within_band
0,high,357,178,0,0.498599,0.000000
1,low,2631,0,0,0.000000,0.000000
2,medium,309,90,135,0.291262,0.436893


In [18]:
# 4d. Friction by root cause category
friction_by_root_cause = (
    case_intel_df.groupby("root_cause_category")
    .agg(
        cases=("consumer_id", "count"),
        false_positives=("is_fp", "sum"),
        borderline_cases=("borderline_flag", "sum")
    )
    .reset_index()
)

friction_by_root_cause["fp_rate_within_category"] = (
    friction_by_root_cause["false_positives"] / friction_by_root_cause["cases"]
)

friction_by_root_cause["borderline_rate_within_category"] = (
    friction_by_root_cause["borderline_cases"] / friction_by_root_cause["cases"]
)

friction_by_root_cause.sort_values("fp_rate_within_category", ascending=False).head(15)


KeyError: 'root_cause_category'

In [17]:
# 4e. Friction by region
friction_by_region = (
    case_intel_df.groupby("latest_delivery_address_region_label")
    .agg(
        cases=("consumer_id", "count"),
        false_positives=("is_fp", "sum"),
        avg_score=("risk_score", "mean")
    )
    .reset_index()
)

friction_by_region["fp_rate_within_region"] = (
    friction_by_region["false_positives"] / friction_by_region["cases"]
)

friction_by_region.sort_values("fp_rate_within_region", ascending=False).head(15)


,latest_delivery_address_region_label,cases,false_positives,avg_score,fp_rate_within_region
39,40.0,3,2,0.417136,0.666667
14,13.0,2,1,0.514091,0.500000
3,2.0,2,1,0.332332,0.500000
37,38.0,30,6,0.216673,0.200000
36,37.0,11,2,0.150286,0.181818
18,17.0,11,2,0.261387,0.181818
20,19.0,11,2,0.354126,0.181818
34,35.0,17,3,0.207628,0.176471
31,32.0,32,5,0.345560,0.156250
35,36.0,15,2,0.236872,0.133333


In [19]:
# 4e. Friction by region
friction_by_region = (
    case_intel_df.groupby("latest_delivery_address_region_label")
    .agg(
        cases=("consumer_id", "count"),
        false_positives=("is_fp", "sum"),
        avg_score=("risk_score", "mean")
    )
    .reset_index()
)

friction_by_region["fp_rate_within_region"] = (
    friction_by_region["false_positives"] / friction_by_region["cases"]
)

friction_by_region.sort_values("fp_rate_within_region", ascending=False).head(15)


,latest_delivery_address_region_label,cases,false_positives,avg_score,fp_rate_within_region
39,40.0,3,2,0.417136,0.666667
14,13.0,2,1,0.514091,0.500000
3,2.0,2,1,0.332332,0.500000
37,38.0,30,6,0.216673,0.200000
36,37.0,11,2,0.150286,0.181818
18,17.0,11,2,0.261387,0.181818
20,19.0,11,2,0.354126,0.181818
34,35.0,17,3,0.207628,0.176471
31,32.0,32,5,0.345560,0.156250
35,36.0,15,2,0.236872,0.133333


In [20]:
# 4f. Likely false-positive review table
likely_fp_cases = case_intel_df[
    (case_intel_df["is_fp"] == 1) | (case_intel_df["borderline_flag"] == 1)
].copy()

likely_fp_cases = likely_fp_cases.sort_values("risk_score", ascending=False)
likely_fp_cases.head(20)


,consumer_id,y_true,risk_score,predicted_label,risk_band,latest_changed_password,max_item_count,days_since_first_transaction,email_address_age_days,email_domain_label,email_domain_tld_label,email_username_length,num_unique_delivery_addresses,latest_delivery_address_name_length,latest_delivery_address_region_label,latest_delivery_address_fraction_vowels,latest_order_amount_usd,latest_item_category,latest_item_product_title,latest_item_quantity,latest_item_tag_count,per_week_purchase_total,per_week_unique_ips,per_week_update_account,per_week_payment_method_change,per_day_add_item_to_cart,per_day_transactions,per_day_payment_method_change,per_day_devices_per_user,per_day_purchase_total,per_day_unique_billing_last4,per_month_logout,per_month_page_activity,per_month_transactions,is_tp,is_fp,is_fn,is_tn,high_transaction_velocity,many_unique_ips,reason_code_combo,borderline_flag
422,804,0,0.999991,1,high,False,24.0,286.341802,380.067298,458.0,17.0,11.0,2.0,15.0,5.0,0.191198,2483.574474,Fish & Seafood,Miso Salmon,1.0,1.0,0.000000,1.0,6.0,4.0,-999.0,5.0,4.0,1.0,0.000000,5.0,-999.0,-999.0,6.481730,0,1,0,0,1,0,high_transaction_velocity,0
674,2479,0,0.999877,1,high,False,8.0,324.359813,3.387109,472.0,17.0,14.0,4.0,13.0,43.0,0.270091,118.576896,Missing,Missing,-999.0,-999.0,-999.000000,6.0,20.0,17.0,-999.0,9.0,12.0,3.0,-999.000000,4.0,-999.0,-999.0,23.780053,0,1,0,0,1,1,"high_transaction_velocity, many_unique_ips",0
2094,7594,0,0.998713,1,high,False,4.0,721.784197,27.678665,1121.0,17.0,13.0,7.0,18.0,10.0,0.188211,69.219536,Carne (Meats),Ropa Vieja,4.0,1.0,536.110000,4.0,12.0,3.0,24.0,9.0,3.0,6.0,536.110000,2.0,1.0,5.0,17.266836,0,1,0,0,1,1,"high_transaction_velocity, many_unique_ips",0
2357,10841,0,0.998429,1,high,False,2.0,425.471962,22.978549,386.0,17.0,9.0,3.0,11.0,33.0,0.328191,109.667419,Shakes & Frozen Custard,Featured Shakes,2.0,1.0,143.910000,2.0,8.0,4.0,6.0,7.0,4.0,3.0,143.910000,2.0,1.0,4.0,4.483389,0,1,0,0,1,0,high_transaction_velocity,0
2720,14652,0,0.998331,1,high,False,2.0,887.493330,117.602970,767.0,17.0,7.0,4.0,14.0,15.0,0.302918,39.965672,Combos,10PC Chicken Nuggets Combo,1.0,1.0,210.865645,5.0,10.0,1.0,23.0,7.0,1.0,8.0,210.865645,2.0,1.0,5.0,6.981462,0,1,0,0,1,1,"high_transaction_velocity, many_unique_ips",0
2578,19471,0,0.997753,1,high,False,2.0,866.629389,16.858432,1121.0,17.0,6.0,1.0,16.0,5.0,0.334848,109.667419,Mains (Dinner),9. Taiwanese Supreme Spicy Hot Soup,1.0,1.0,-999.000000,2.0,8.0,5.0,4.0,10.0,5.0,4.0,-999.000000,3.0,2.0,3.0,6.503263,0,1,0,0,1,0,high_transaction_velocity,0
706,6570,0,0.997502,1,high,False,1.0,146.032724,15.695816,472.0,17.0,10.0,2.0,10.0,36.0,0.596274,266.122240,Wraps,Falafel (Vegetarian),1.0,1.0,44.469832,4.0,9.0,2.0,-999.0,-999.0,0.0,1.0,0.000000,-999.0,1.0,4.0,4.483389,0,1,0,0,0,1,many_unique_ips,0
2918,3350,0,0.997273,1,high,False,1.0,314.989689,54.104023,458.0,17.0,17.0,12.0,15.0,44.0,0.420974,192.570729,Specialty Subs,Italian Sub (Large),1.0,1.0,5.500000,2.0,17.0,2.0,35.0,12.0,2.0,3.0,5.500000,-999.0,1.0,5.0,7.807731,0,1,0,0,1,0,high_transaction_velocity,0
448,17243,0,0.996711,1,high,False,4.0,162.253757,0.002994,386.0,17.0,12.0,2.0,10.0,5.0,0.550422,703.591640,Tequila,Patrón Silver Tequila | 750ml 40% abv,2.0,1.0,-999.000000,4.0,17.0,3.0,-999.0,6.0,3.0,4.0,-999.000000,3.0,-999.0,-999.0,6.793893,0,1,0,0,1,1,"high_transaction_velocity, many_unique_ips",0
246,3668,0,0.995258,1,high,False,2.0,9.592393,346.053120,386.0,17.0,14.0,1.0,16.0,33.0,0.460064,136.607412,Coffee & Espresso,Freshly Brewed Coffee Hot,1.0,1.0,64.140000,3.0,2.0,0.0,4.0,1.0,0.0,3.0,27.820000,1.0,9.0,5.0,6.966650,0,1,0,0,0,1,many_unique_ips,0


# 4. Save Data

In [ ]:
# cases_by_risk_band.to_csv("datasets/cases_by_risk_band.csv", index=False)
# cases_by_root_cause.to_csv("datasets/cases_by_root_cause.csv", index=False)
# top_product_categories_flagged.to_csv("datasets/top_product_categories_flagged.csv", index=False)
# fp_by_region.to_csv("datasets/false_positives_by_region.csv", index=False)
# fp_by_category.to_csv("datasets/false_positives_by_category.csv", index=False)
# reason_code_combos.to_csv("datasets/reason_code_combos.csv", index=False)
# friction_by_risk_band.to_csv("datasets/friction_by_risk_band.csv", index=False)
# friction_by_root_cause.to_csv("datasets/friction_by_root_cause.csv", index=False)
# friction_by_region.to_csv("datasets/friction_by_region.csv", index=False)
